[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_92_Regression_Canary_Evals_and_Scoring_Drift.ipynb)

# Lesson 92 — Regression & Canary Evals in CI + Scoring Drift
### Phase 11: Evaluation & Trust at Scale · Lesson 2 of 6

In **Lesson 91** you built the foundations of trust: a **golden dataset**, an **LLM‑as‑judge**, four RAG metrics from scratch, and an `evaluate()` **scoreboard**. That scoreboard answered one question: *how good is my system **right now**?*

But you don't ship a system once. You ship it **every day** — a new prompt, a bigger `k`, a swapped embedder, a reranker tweak. Each change can silently make things **worse**. A scoreboard that is green *today* proves nothing about *tomorrow's* commit.

> **This lesson turns the scoreboard into a GATE.** A gate runs on **every change**, compares against a **known‑good baseline**, and **fails the build** when quality regresses — *before* the change reaches users.

| Lesson | Topic | Status |
|---|---|---|
| L91 | Golden datasets + LLM‑as‑judge + 4 metrics + scoreboard | ✅ |
| **L92** | **Regression & canary evals in CI + scoring drift** | **← you are here** |
| L93 | Human‑in‑the‑loop feedback → data flywheel | ⏳ |
| L94 | Red‑teaming & safety evals (jailbreaks, prompt injection) | ⏳ |
| L95 | Cost / latency budgets + load‑testing the `/ask` path | ⏳ |
| L96 | Phase‑11 capstone: reusable `agent-evals` harness | ⏳ |

**Four ideas you'll build today**

1. **Baselines + tolerances** — floors *and* relative drops, and why you need **both**.
2. **Per‑metric & per‑row regression detection** — the aggregate lies; localize the break.
3. **Canary rows** — a handful of critical behaviors that must **never** regress (zero tolerance).
4. **Scoring drift** — the subtle killer: the **judge itself moves**, so your scores change even when the system didn't. A **frozen calibration set** catches it.

Runs **fully offline and keyless** — deterministic Python, no API key needed.

## §0 · Why one green scoreboard isn't enough

Picture your CI pipeline. A teammate opens a pull request that "improves" chunking. Tests pass (the code runs). It merges. Two days later, support tickets spike: the bot now confidently answers questions it should refuse.

What went wrong? **Unit tests check that code *runs*. They don't check that the agent is still *good*.** Quality lives in fuzzy, statistical metrics — faithfulness, correctness, recall — not in `assert x == 5`.

A **regression gate** closes that hole. On every change it:

1. Re‑runs the golden‑set evaluation (L91) → today's scores.
2. Compares them to a stored **baseline** (last‑known‑good scores).
3. **Blocks the merge** if any metric dropped too far, or any *canary* behavior broke, or the *judge itself drifted*.

Three distinct failure signals, three distinct mechanisms. Let's build them one at a time on top of a compact version of L91's harness.

In [ ]:
# L92 is fully offline & keyless. numpy is the only dep and Colab already ships it.
!pip install numpy -q
print("ready")

## §1 · A compact eval harness (recap of L91)

To keep this lesson self‑contained we rebuild a *tiny* version of L91's pieces:

- a **content‑word tokenizer** + **bag‑of‑words cosine** (deterministic stand‑ins for embeddings),
- a 5‑doc **espresso corpus** + a 5‑row **golden set** (including one **out‑of‑domain refusal row**, `g5`),
- a **system‑under‑test** = retriever + generator with a **relevance gate** (refuse when nothing scores),
- an `evaluate()` that returns **per‑row** and **aggregate** scores for two metrics: `correct` and `faith` (faithfulness).

The system takes an optional **`sabotage`** argument so we can inject *known regressions* on demand, and the judge takes a **`judge_bias`** so we can simulate *scoring drift* later. Everything below is deterministic.

In [ ]:
import re, math
from collections import Counter

# ---------- tiny deterministic text primitives (from L91) ----------
STOP = set("a an the of to for is are be with your you it this that on in and or do i my what should at so how".split())

def toks(s):
    # content-word tokenizer: lowercase alphanum words, minus stopwords
    return [w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in STOP]

def bow(s):
    return Counter(toks(s))

def cosine(a, b):
    # cosine similarity over two bag-of-words counters (stand-in for dense embeddings)
    dot = sum(a[w]*b[w] for w in a if w in b)
    na = math.sqrt(sum(v*v for v in a.values())); nb = math.sqrt(sum(v*v for v in b.values()))
    return dot/(na*nb) if na and nb else 0.0

def contained(claim, context):
    # fraction of the claim's content words that appear in the context (grounding proxy)
    ct = toks(claim); cx = set(toks(context))
    return sum(1 for w in ct if w in cx)/len(ct) if ct else 0.0

# ---------- the knowledge base ----------
CORPUS = {
 "d_grind": "Bitter espresso usually means over-extraction. Grind coarser or shorten the shot time.",
 "d_sour":  "Sour espresso means under-extraction. Grind finer or raise the water temperature.",
 "d_crema": "Thin crema is caused by stale beans. Use beans roasted within the last three weeks.",
 "d_temp":  "The ideal brew temperature is between 90 and 96 degrees Celsius for most beans.",
 "d_dose":  "A standard double shot uses 18 grams of ground coffee for a 36 gram yield.",
}

# ---------- golden set: question, ground-truth phrase, gold doc, must-refuse flag ----------
GOLD = [
 {"id":"g1","q":"why is my espresso bitter",          "ans":"over-extraction", "doc":"d_grind","refuse":False},
 {"id":"g2","q":"my shot tastes sour what do i do",   "ans":"under-extraction","doc":"d_sour", "refuse":False},
 {"id":"g3","q":"why is the crema so thin",           "ans":"stale beans",     "doc":"d_crema","refuse":False},
 {"id":"g4","q":"what temperature should i brew at",  "ans":"90 and 96",       "doc":"d_temp", "refuse":False},
 {"id":"g5","q":"is the moon made of cheese",         "ans":"",                "doc":None,     "refuse":True},
]

# ---------- system under test = retriever + generator + relevance gate ----------
def retrieve(q, corpus):
    qb = bow(q)
    return sorted(((d, cosine(qb, bow(t))) for d,t in corpus.items()), key=lambda x:-x[1])

def answer(q, corpus, sabotage=None):
    # sabotage: dict of injected regressions used to demo the gate. None = healthy system.
    sab = sabotage or {}
    if sab.get("hallucinate_q") == q:
        # regression: model answers an out-of-domain question instead of refusing
        return {"text":"Yes, absolutely — that is correct.", "ctx":"", "doc":None, "refused":False}
    work = dict(corpus)
    if sab.get("drop_doc"):
        work.pop(sab["drop_doc"], None)   # simulate a bad index change: a doc vanishes
    top_doc, top_score = retrieve(q, work)[0]
    if top_score < 0.05:                  # relevance gate -> refuse when nothing is relevant
        return {"text":"I don't have information on that.", "ctx":"", "doc":None, "refused":True}
    ctx = work[top_doc]
    return {"text":ctx, "ctx":ctx, "doc":top_doc, "refused":False}

# ---------- the judge: scores one row. judge_bias models SCORING DRIFT (added later) ----------
def score_row(row, out, judge_bias=0.0):
    if row["refuse"]:
        correct = 1.0 if out["refused"] else 0.0
        faith   = 1.0 if out["refused"] else 0.0   # a correct refusal is perfectly faithful
    else:
        correct = 1.0 if row["ans"] and row["ans"] in out["text"] else 0.0
        faith   = max(0.0, min(1.0, contained(out["text"], out["ctx"]) + judge_bias))
    return {"correct":round(correct,3), "faith":round(faith,3)}

def evaluate(corpus, gold, sabotage=None, judge_bias=0.0):
    rows = {r["id"]: score_row(r, answer(r["q"], corpus, sabotage), judge_bias) for r in gold}
    agg  = {m: round(sum(rows[i][m] for i in rows)/len(rows), 3) for m in ("correct","faith")}
    return {"rows":rows, "agg":agg}

demo = evaluate(CORPUS, GOLD)
print("aggregate:", demo["agg"])
for rid, sc in demo["rows"].items(): print(" ", rid, sc)

## §2 · The baseline — capture "last known good"

A regression is only meaningful **relative to a reference point**. That reference is the **baseline**: the scoreboard from your last release you were happy with, **committed to the repo** (e.g. `eval/baseline.json`) right next to the code.

> The baseline is a *contract*: "we promise the system is at least this good." Every future run is judged against it. When you make a **deliberate** improvement, you re‑bless a new baseline; you never let it drift silently.

On our healthy system the baseline is a clean sweep — `correct = 1.0`, `faith = 1.0` — and, crucially, the **per‑row** scores too (we'll need those for canaries).

In [ ]:
import json, copy

baseline = evaluate(CORPUS, GOLD)          # healthy system -> last known good
BASELINE = copy.deepcopy(baseline)         # this is what you'd commit as eval/baseline.json

print("BASELINE (committed to the repo):")
print(json.dumps(BASELINE, indent=2))
# 💡 EXPERIMENT: change a corpus doc so a gold phrase disappears, re-run -> watch the
#    baseline itself drop. That is your signal you're about to bless a *worse* baseline.

## §3 · Thresholds — floors **and** tolerances (you need both)

Two different questions, two different guards:

**Floor** — *"is quality still acceptable in absolute terms?"* A hard minimum, e.g. faithfulness must be ≥ 0.90 no matter what. Floors stop you shipping something bad even if the baseline was already low.

**Tolerance** — *"did this change make things meaningfully worse than before?"* A relative guard: fail if the metric dropped more than, say, 0.05 **below the baseline** — even if it's still above the floor.

Why both? Consider faithfulness with `floor=0.90`, baseline `0.99`:

- A drop `0.99 → 0.93` clears the floor, but a **0.06 erosion** slips through with a floor alone → **tolerance catches it.**
- A system that was *always* mediocre (baseline `0.91`) then dips to `0.905` clears the tolerance, but sits at the edge of unacceptable → **the floor is your backstop.**

Floors catch *absolute* badness; tolerances catch *slow erosion*. Real pipelines encode both.

In [ ]:
# per-metric config: hard floor + how far below baseline we tolerate before failing
THRESHOLDS = {
  "correct": {"floor":0.80, "tol":0.10},
  "faith":   {"floor":0.90, "tol":0.05},
}

def check_metric(name, cur, base, cfg):
    # returns a list of human-readable failure reasons ([] = this metric passed)
    reasons = []
    if cur < cfg["floor"]:
        reasons.append(f"{name}={cur:.3f} BELOW FLOOR {cfg['floor']:.2f}")
    if cur < base - cfg["tol"]:
        reasons.append(f"{name} dropped {base-cur:.3f} vs baseline {base:.3f} (tol {cfg['tol']:.2f})")
    return reasons

# sanity: the healthy system violates neither guard
fails = []
for m,cfg in THRESHOLDS.items():
    fails += check_metric(m, baseline["agg"][m], BASELINE["agg"][m], cfg)
print("healthy-system metric failures:", fails or "NONE (as expected)")

## §4 · The aggregate lies — detect regressions **per row**

Averages hide localized damage. If one of five rows breaks completely, `correct` only falls `1.0 → 0.8` — easy to wave away as noise. But that "noise" might be your **refusal** behavior collapsing.

So a good gate checks **two levels**:

1. **Aggregate vs baseline** (floors + tolerances from §3) — the headline.
2. **Per‑row vs baseline** — which *specific* row got worse, so you can debug in seconds instead of bisecting.

Let's inject a realistic regression: a chunking change accidentally **drops `d_crema` from the index**. Watch the aggregate move a little while the per‑row view names the culprit exactly.

In [ ]:
def per_row_regressions(cur_rows, base_rows):
    # returns list of (row_id, metric, base_value, cur_value) for every metric that dropped
    out = []
    for rid in base_rows:
        for m in base_rows[rid]:
            if cur_rows[rid][m] < base_rows[rid][m]:
                out.append((rid, m, base_rows[rid][m], cur_rows[rid][m]))
    return out

regressed = evaluate(CORPUS, GOLD, sabotage={"drop_doc":"d_crema"})
print("aggregate now :", regressed["agg"], "   (baseline:", BASELINE["agg"], ")")
print("\nper-metric aggregate check:")
for m,cfg in THRESHOLDS.items():
    print(" ", m, check_metric(m, regressed["agg"][m], BASELINE["agg"][m], cfg) or "ok")
print("\nper-row regressions (the aggregate hid WHERE):")
for r in per_row_regressions(regressed["rows"], BASELINE["rows"]):
    print("  row", r[0], r[1], f"{r[2]} -> {r[3]}")

## §5 · Canary rows — behaviors that must **never** regress

Some behaviors are non‑negotiable. Your bot **must** refuse "is the moon made of cheese." It **must** cite a source for a medical claim. These aren't "nice on average" — a single failure is a headline.

A **canary** is a golden row (or a metric on it) marked **zero‑tolerance**: if it drops *at all* versus baseline, the build fails, full stop. Canaries don't get averaged away — they're the smoke detector wired straight to the alarm.

> Named after canaries in coal mines: cheap, sensitive, and their distress is an unambiguous *stop*. Keep the canary set **small and sacred** — every row in it is a promise you've decided you will never quietly break.

We'll mark `g5` (the refusal) and `g3` (a known‑fragile retrieval) as canaries, then show they catch two different regressions the aggregate would happily average away.

In [ ]:
# canary map: row_id -> the metrics on it that must never drop below baseline
CANARIES = {"g5":["correct","faith"], "g3":["correct"]}

def canary_check(cur_rows, base_rows):
    fails = []
    for rid, metrics in CANARIES.items():
        for m in metrics:
            if cur_rows[rid][m] < base_rows[rid][m]:
                fails.append(f"CANARY {rid}.{m}: {base_rows[rid][m]} -> {cur_rows[rid][m]}")
    return fails

# regression A: dropped d_crema (from §4) trips the g3 canary
a = evaluate(CORPUS, GOLD, sabotage={"drop_doc":"d_crema"})
print("A) drop d_crema  ->", canary_check(a["rows"], BASELINE["rows"]))

# regression B: model stops refusing the out-of-domain question -> g5 canary fires
b = evaluate(CORPUS, GOLD, sabotage={"hallucinate_q":"is the moon made of cheese"})
print("B) refuse->hallucinate ->", canary_check(b["rows"], BASELINE["rows"]))
print("   (note B's aggregate:", b["agg"], "- 'correct' only fell 1.0->0.8, easy to ignore without a canary)")

## §6 · Scoring drift — when the **meter** moves, not the system

Here is the subtle one that burns teams. Your scores can change **without the system changing at all** — because the thing *measuring* it changed:

- your **LLM judge** got a silent model update (`gpt‑4o‑2024‑05` → `‑08`),
- someone tweaked the **judge prompt** or bumped its temperature,
- your **embedding model** version rolled forward under the hood.

Now every number shifts. If the judge got more lenient, a *real* regression can be **masked** (scores stay green). If it got stricter, you'll chase **phantom** regressions that aren't in your system at all. Either way, **you can no longer trust the ruler.**

**The fix: a frozen calibration set.** A handful of `(answer, context)` pairs with a **pinned expected judge score**, committed to the repo. These inputs *never change*. Re‑score them every run. If the judge's output on this frozen set moves, it's the **meter** that drifted — independent of your system‑under‑test.

We model drift with `judge_bias`: a shift added to the judge's faithfulness score. Watch the punchline — with drift on, the **main eval still looks perfect** (scores were already at the ceiling), yet the calibration set exposes the drift immediately.

In [ ]:
# frozen calibration set: fixed (answer, ctx) pairs + the judge score they SHOULD get.
# these are committed and never edited. their job is to audit the judge, not the system.
CALIB = [
 {"id":"c1","answer":"grind coarser and lower temperature","ctx":"Grind coarser or shorten the shot time.","pin":0.5},
 {"id":"c2","answer":"use fresh beans roasted recently",   "ctx":"Use beans roasted within the last three weeks.","pin":0.6},
]

def judge_faith(ans, ctx, judge_bias=0.0):
    return round(max(0.0, min(1.0, contained(ans, ctx) + judge_bias)), 3)

def calibration_drift(judge_bias=0.0, eps=0.03):
    rep, drift = [], False
    for c in CALIB:
        got = judge_faith(c["answer"], c["ctx"], judge_bias)
        d = abs(got - c["pin"]); drift = drift or d > eps
        rep.append((c["id"], c["pin"], got, round(d,3)))
    return drift, rep

# healthy judge: calibration matches the pins exactly
print("no drift  (judge_bias=0.00):", calibration_drift(0.0))
# judge silently gets more lenient (+0.15) with the SAME healthy system:
main = evaluate(CORPUS, GOLD, judge_bias=0.15)
print("\nWITH judge drift +0.15 -> main eval agg:", main["agg"], "  <- STILL looks perfect!")
print("calibration set catches it:", calibration_drift(0.15))
# 💡 EXPERIMENT: try judge_bias=-0.2 (a stricter judge). The main eval faith DROPS even
#    though the system is unchanged -> a PHANTOM regression. Calibration tells you it's the meter.

## §7 · Assemble the gate + wire it into CI

Now fold all three signals into one `regression_gate()` that returns `ok` plus a structured report:

1. **per‑metric** floors + tolerances (§3),
2. **canary** rows, zero tolerance (§5),
3. **scoring drift** via the frozen calibration set (§6).

`ok` is `True` only if **all three** pass. In CI you exit non‑zero when `ok` is `False` so the pipeline **blocks the merge** — exactly like the `ci.yml` you shipped in **L90**, but now the test being run is your *quality* gate, not just `pytest`.

In [ ]:
def regression_gate(corpus, gold, baseline, sabotage=None, judge_bias=0.0):
    cur = evaluate(corpus, gold, sabotage, judge_bias)
    rep = {"agg":cur["agg"], "metric_fail":[], "canary_fail":[], "drift":None}
    for m,cfg in THRESHOLDS.items():                                  # 1) floors + tolerances
        rep["metric_fail"] += check_metric(m, cur["agg"][m], baseline["agg"][m], cfg)
    rep["canary_fail"] = canary_check(cur["rows"], baseline["rows"])  # 2) canaries
    drift, drift_rep = calibration_drift(judge_bias)                  # 3) scoring drift
    rep["drift"] = drift_rep if drift else None
    rep["ok"] = not (rep["metric_fail"] or rep["canary_fail"] or drift)
    return rep

def show(title, rep):
    print(f"=== {title} ===")
    print("  agg        :", rep["agg"])
    print("  metric_fail:", rep["metric_fail"] or "none")
    print("  canary_fail:", rep["canary_fail"] or "none")
    print("  drift      :", rep["drift"] or "none")
    print("  GATE       :", "PASS ✅" if rep["ok"] else "FAIL ❌ (block merge, exit 1)\n")

show("clean commit",            regression_gate(CORPUS, GOLD, BASELINE))
show("dropped d_crema (index)", regression_gate(CORPUS, GOLD, BASELINE, sabotage={"drop_doc":"d_crema"}))
show("stopped refusing (g5)",   regression_gate(CORPUS, GOLD, BASELINE, sabotage={"hallucinate_q":"is the moon made of cheese"}))
show("judge drift +0.15",       regression_gate(CORPUS, GOLD, BASELINE, judge_bias=0.15))

### The CI wiring (callback to L90)

In L90 you shipped a service with a `ci.yml` that ran `pytest` on every push. The regression gate is *another job* in that same workflow — it runs the golden‑set evaluation and **exits non‑zero on any regression**, so GitHub blocks the merge. The Python entry point is three lines; the YAML is the L90 pattern with the command swapped.

In [ ]:
# eval/gate.py  -- the CI entry point (exits 1 on regression so the pipeline fails)
GATE_PY = '''
import json, sys
from eval.harness import CORPUS, GOLD, evaluate, regression_gate   # your committed harness
BASELINE = json.load(open("eval/baseline.json"))                    # last known good
rep = regression_gate(CORPUS, GOLD, BASELINE)
print(json.dumps(rep, indent=2))
sys.exit(0 if rep["ok"] else 1)                                     # non-zero => block merge
'''

# .github/workflows/ci.yml -- same shape as L90, new step: the quality gate
CI_YML = '''
name: ci
on: [push, pull_request]
jobs:
  eval-gate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - run: pip install -e . numpy
      - name: Regression + canary + drift gate
        run: python -m eval.gate          # exits 1 on regression -> PR blocked
'''
print(GATE_PY); print(CI_YML)
print("This is the L90 ci.yml pattern with the test swapped from pytest to the quality gate.")

## §8 · Ten pitfalls

1. **No baseline in the repo.** "Compare to last time" is meaningless if *last time* isn't committed. Version `baseline.json` next to the code.
2. **Floors only, no tolerance.** Slow erosion (0.99→0.94→0.91) never trips a floor until it's a crisis. Add relative tolerances.
3. **Tolerance only, no floor.** A system that was always mediocre keeps sliding within tolerance each PR. Keep an absolute backstop.
4. **Aggregate‑only checks.** One broken row hides in the mean. Always add per‑row (or per‑slice) detection.
5. **Canary set too big.** If everything is a canary, nothing is — flaky rows will block every PR and people disable the gate. Keep it small and sacred.
6. **Ignoring scoring drift.** A judge/embedder update silently rescales everything. Without a frozen calibration set you can't tell system regressions from meter drift.
7. **Un‑pinned judge.** Calling `gpt‑4o` (floating alias) as your judge = a moving ruler by design. Pin the exact model + prompt + temperature; snapshot outputs.
8. **Re‑blessing baselines carelessly.** Auto‑updating the baseline to "whatever ran today" launders regressions into the new normal. Re‑bless only on a *deliberate, reviewed* improvement.
9. **Tiny golden set → noisy gate.** With 5 rows one flip is 20%. Grow the set so tolerances are statistically meaningful (L93's flywheel feeds this).
10. **Non‑deterministic eval, no seed.** If the same commit yields different scores run‑to‑run, you can't attribute a drop to the change. Fix seeds / temperature=0 for the eval path.

## §9 · Verification checklist

Deterministic assertions covering every mechanism we built. All must print **PASS**.

In [ ]:
checks = []
def ck(name, cond): checks.append((name, bool(cond))); print(("PASS" if cond else "FAIL"), "-", name)

# 1) baseline is a clean sweep on the healthy system
ck("baseline agg correct==1.0 and faith==1.0", BASELINE["agg"]["correct"]==1.0 and BASELINE["agg"]["faith"]==1.0)
# 2) clean commit passes the full gate
ck("clean commit -> gate PASS", regression_gate(CORPUS, GOLD, BASELINE)["ok"])
# 3) dropping d_crema is caught, and localized to row g3
rr = evaluate(CORPUS, GOLD, sabotage={"drop_doc":"d_crema"})
ck("drop d_crema -> gate FAIL", not regression_gate(CORPUS, GOLD, BASELINE, sabotage={"drop_doc":"d_crema"})["ok"])
ck("drop d_crema -> per-row names g3", any(r[0]=="g3" for r in per_row_regressions(rr["rows"], BASELINE["rows"])))
ck("drop d_crema -> g3 canary fires", any("g3" in f for f in canary_check(rr["rows"], BASELINE["rows"])))
# 4) refusal collapse is caught by the g5 canary
hb = evaluate(CORPUS, GOLD, sabotage={"hallucinate_q":"is the moon made of cheese"})
ck("stop refusing -> g5 canary fires", any("g5" in f for f in canary_check(hb["rows"], BASELINE["rows"])))
# 5) THE drift punchline: main eval invisible, calibration visible
ck("judge drift +0.15 -> main eval faith still 1.0 (invisible)", evaluate(CORPUS, GOLD, judge_bias=0.15)["agg"]["faith"]==1.0)
ck("judge drift +0.15 -> calibration flags drift", calibration_drift(0.15)[0])
ck("judge drift +0.15 -> gate FAIL via drift", not regression_gate(CORPUS, GOLD, BASELINE, judge_bias=0.15)["ok"])
ck("no drift -> calibration clean", not calibration_drift(0.0)[0])
# 6) floors AND tolerances both usable
ck("check_metric flags below-floor", len(check_metric("faith", 0.80, 1.0, THRESHOLDS["faith"]))>0)
ck("check_metric flags over-tolerance drop", any("dropped" in r for r in check_metric("faith", 0.94, 1.0, THRESHOLDS["faith"])))

assert all(v for _,v in checks), "some checks FAILED"
print("\nALL", len(checks), "CHECKS PASS ✅")

## §10 · Recap, homework & what's next

**What you built.** You promoted L91's one‑shot scoreboard into a **regression gate** that runs on every change and blocks bad merges — with three independent guards:

- **Baselines + floors + tolerances** — absolute badness *and* slow erosion, both caught.
- **Per‑row + canary detection** — the aggregate can't hide a broken behavior; canaries make critical rows zero‑tolerance.
- **Scoring‑drift detection** — a **frozen calibration set** audits the *judge itself*, so you never confuse a moving ruler for a moving system.

And you wired it into CI as an `eval-gate` job (the L90 `ci.yml` pattern), exiting non‑zero on any regression.

**Homework**

1. Add a `slice` field to golden rows (e.g. `retrieval` vs `refusal`) and report per‑slice aggregates — often a regression is confined to one slice.
2. Make `regression_gate` emit a **GitHub‑style step summary** (a markdown table of pass/fail) instead of raw dicts.
3. Add a **"blessing" script** that only updates `baseline.json` when a human passes `--approve`, and prints a diff of every metric that moved.
4. Turn the 5‑row golden set into 20 rows and re‑tune tolerances so one flaky row can't fail the build.
5. Pin a real judge: snapshot an LLM judge's raw scores on the calibration set to a file, and diff against it each run.

**Next lesson — L93: Human‑in‑the‑loop feedback → the data flywheel.** Your golden set is small, so your gate is noisy (pitfall #9). L93 closes the loop: capture real user thumbs‑up/down and corrections, triage them, and **promote the good ones into new golden + canary rows** — the engine that grows the very datasets this gate depends on.